# Model Development — Robust Multi-Objective Distribution Network Design

**Case study:** Personal care manufacturer distribution network, Java, Indonesia.
**Scope:** Geography = Java only. Products = Beauty + Personal Care segments only.
**Data:** cleaned in `Validation.ipynb`, files under `D:\KLTN\Data Dictionary\Model - Data used\`.

This notebook is structured as a report chapter: each section has a narrative part (problem statement, notation, math model, method) for you to write, followed by the code that implements it.


---
# 1. Problem Statement

> ✍️ **Write here:** the research problem in plain language — what network decisions are being made (which facilities stay open, how goods flow), what makes it multi-objective (cost vs. coverage vs. CO2), why demand uncertainty matters, and why a robust optimization + AUGMECON2 + TOPSIS approach was chosen. This is the narrative version of what you already drafted for your advisor.

*(placeholder — fill in)*


---
# 2. Notation

## 2.1 Sets and Indices

| Symbol | Description | Source |
|---|---|---|
| $j \in J$ | Facilities (FIXED $\cup$ DECISION) | `FacilityMaster.parquet` |
| $k \in K$ | Demand clusters (ShipToGroupID) | `shipto_crosswalk.parquet` |
| $t \in T$ | Time period (month), if time-indexed | `b2b_so_final.parquet` |

> ✍️ **Write here:** add/adjust any other sets you introduce (e.g. product groups, scenarios).

## 2.2 Parameters

| Symbol | Description | Source |
|---|---|---|
| $\bar d_k$ | Nominal (mean) monthly demand at cluster $k$, in pallets | Section 4.2 |
| $\hat d_k$ | Demand deviation at cluster $k$ (uncertainty budget) | Section 4.2 |
| $Dist_{jk}$ | Distance, facility $j$ to cluster $k$ | Section 4.4 |
| $D_{max}$ | Max SLA distance (150 km, Java B2B) | Paragon Assumption Doc §4.7 |
| $CostLastMile_{jk}$ | Last-mile delivery cost, IDR/pallet | `ENO_CostLastMileB2B.parquet` |
| $CostTransfer_{jj'}$ | Facility-to-facility transfer cost, IDR/pallet | `ENO_CostTransfer.parquet` |
| $CostSupply_{fj}$ | Factory-to-facility supply cost, IDR/pallet | `ENO_CostSupply.parquet` |
| $DOS_{j,pg}$ | Days of stock, facility × product group | `ENO_DOS.parquet` |
| $WACC$ | Cost of capital for IOC (10%) | Model.md, confirmed by Paragon |
| $CO2_{jk}$ | Emissions factor, facility → cluster | `Data Dictionary/CO2/` |

> ✍️ **Write here:** any additional parameters (capacity, fixed cost, etc.) as you finalize the formulation.

## 2.3 Decision Variables

| Symbol | Type | Description |
|---|---|---|
| $Y_j$ | Binary | 1 if facility $j$ (DECISION set) is open |
| $X_{jk}$ | Continuous $\ge 0$ | Flow (pallets) from facility $j$ to cluster $k$ |
| $U_k$ | Binary/continuous | Unmet/uncovered demand indicator at cluster $k$ |

> ✍️ **Write here:** finalize exact variable set once the constraint structure is settled.


---
# 3. Mathematical Model

## 3.1 Objective Functions

**$f_1$ — Total Cost (minimize)**

$$
f_1 = \sum_{j,k} CostLastMile_{jk} \cdot X_{jk} + \sum_{j,j'} CostTransfer_{jj'} \cdot X_{jj'} + \sum_{f,j} CostSupply_{fj} \cdot X_{fj} + \sum_j FixedStorageCost_j \cdot Y_j + IOC
$$

> ✍️ **Write here:** finalize the exact cost terms (which are linear in $X$ vs. fixed per $Y_j$), and the IOC formula (uses $DOS$ and $WACC$).

**$f_2$ — Uncovered Demand (minimize)**

$$
f_2 = \sum_k \bar d_k \cdot U_k
$$

**$f_3$ — CO2 Emissions (minimize)**

$$
f_3 = \sum_{j,k} CO2_{jk} \cdot X_{jk}
$$

> ✍️ **Write here:** confirm emissions factor source/units (Global Logistics Emissions Council Framework).

## 3.2 Constraints

> ✍️ **Write here** each constraint in LaTeX as you finalize it. Skeleton below — fill in:

**Flow balance**
$$
\sum_j X_{jk} = \bar d_k \quad \forall k
$$

**Coverage**
$$
U_k \le 1 - \sum_{j:\, Dist_{jk} \le D_{max}} Y_j \quad \forall k
$$

**Robust demand constraint (Bertsimas & Sim, 2004)**
$$
\text{(write the budget-of-uncertainty reformulation here, using } \hat d_k \text{ and } \Gamma \text{)}
$$

**Facility status**
$$
Y_j = 1 \quad \forall j \in \text{FIXED (NDC)}
$$

**Domain**
$$
Y_j \in \{0,1\}, \quad X_{jk} \ge 0
$$


---
# 4. Data Preparation

## 4.1 Load Cleaned Data

| File | Contents |
|---|---|
| `customer_master_clean.parquet` | Customer ship-to locations (Java only) |
| `b2b_so_final.parquet` | B2B sales orders Jan–Nov 2025 (Java + Beauty/Personal Care) |
| `shipto_crosswalk.parquet` | `ShipToID` → `ShipToGroupID` mapping |
| `product_master_clean.parquet` | Product master (Beauty+Personal Care, `KGPerPallet` fixed) |
| `FacilityMaster.parquet` | 27 facilities in Java |


In [1]:
# ===== Import libraries =====
import pandas as pd
import numpy as np
import os

# ===== Paths =====
DATA_PATH = r"D:\KLTN\Data Dictionary\Model - Data used"

# Root chứa raw scenario data của CEL (Baseline Constrained + Unconstrained/Greenfield)
# Dùng riêng cho các bảng cost ở Section 4.5/4.6 - xem note chọn scenario ở đó
RAW_DATA_PATH = r"D:\KLTN\Data Dictionary\Modeling\Data"

# ===== Load các bảng nhỏ (customer/shipto/product/facility) =====
customer_master = pd.read_parquet(os.path.join(DATA_PATH, "customer_master_clean.parquet"))
shipto_crosswalk = pd.read_parquet(os.path.join(DATA_PATH, "shipto_crosswalk.parquet"))
product_master = pd.read_parquet(os.path.join(DATA_PATH, "product_master_clean.parquet"))
facility_master = pd.read_parquet(os.path.join(DATA_PATH, "FacilityMaster.parquet"))

# ===== Load b2b_so (lấy hết 27 cột cho chắc, ko lọc cột nữa) =====
b2b_so = pd.read_parquet(os.path.join(DATA_PATH, "b2b_so_final.parquet"))

# ===== Giảm RAM cho b2b_so bằng cách convert cột lặp lại nhiều sang dtype "category" =====
# Ko bớt cột, nhưng vẫn tối ưu được RAM cho các cột có giá trị lặp lại rất nhiều
# (ShipToID/ProductID chỉ có vài nghìn giá trị unique trên 14.5 triệu dòng,
#  OrderStatus chỉ có vài giá trị như Done/Canceled) -> convert category ko mất thông tin gì
b2b_so["ShipToID"] = b2b_so["ShipToID"].astype("category")
b2b_so["ProductID"] = b2b_so["ProductID"].astype("category")
b2b_so["OrderStatus"] = b2b_so["OrderStatus"].astype("category")

## 4.2 Demand Aggregation ($\bar d_k$, $\hat d_k$)

1. Merge `b2b_so` with `shipto_crosswalk` on `ShipToID` → attach `ShipToGroupID`.
2. Merge with `product_master` on `ProductID` → get `KGPerPallet` → compute `PalletQty`.
3. Aggregate `PalletQty` by `ShipToGroupID` × month (11 months).
4. Compute $\bar d_k$ (mean) and $\hat d_k$ (deviation) per cluster.


In [2]:
# TODO: merge b2b_so -> shipto_crosswalk -> product_master, compute PalletQty
# TODO: aggregate by ShipToGroupID x month
# TODO: compute d_bar (mean) and d_hat (deviation) per ShipToGroupID

# ===== BƯỚC 1: Map ShipToID -> ShipToGroupID (demand node của model) =====
b2b_agg = b2b_so.merge(shipto_crosswalk, on="ShipToID", how="left")

# ===== BƯỚC 2: Convert kg -> pallet bằng KGPerPallet của từng product =====
# Demand lấy hết mọi order, kể cả Canceled - ko lọc OrderStatus
b2b_agg = b2b_agg.merge(product_master[["ProductID", "KGPerPallet"]], on="ProductID", how="left")
b2b_agg["QtyInPallet"] = b2b_agg["QtyOrderedInKG"] / b2b_agg["KGPerPallet"]

# ===== BƯỚC 3: Lấy tháng từ OrderedTimestamp để tính demand theo tháng =====
b2b_agg["OrderMonth"] = b2b_agg["OrderedTimestamp"].dt.to_period("M")

# ===== BƯỚC 4: Aggregate demand (pallet) theo (ShipToGroupID, tháng) =====
monthly_demand = b2b_agg.groupby(["ShipToGroupID", "OrderMonth"])["QtyInPallet"].sum().reset_index()

# ===== BƯỚC 5: Điền đủ 0 cho node/tháng ko có order (0 là data point thật, ko phải thiếu) =====
all_nodes = shipto_crosswalk["ShipToGroupID"].unique()
all_months = monthly_demand["OrderMonth"].unique()
full_index = pd.MultiIndex.from_product([all_nodes, all_months], names=["ShipToGroupID", "OrderMonth"])
monthly_demand_full = (
    monthly_demand.set_index(["ShipToGroupID", "OrderMonth"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# ===== BƯỚC 6: Tính d_bar (nominal) và d_hat (deviation) cho từng node =====
demand_agg = monthly_demand_full.groupby("ShipToGroupID")["QtyInPallet"].agg(
    d_bar="mean",
    d_max="max"
).reset_index()
demand_agg["d_hat"] = (demand_agg["d_max"] - demand_agg["d_bar"]).clip(lower=0)

# ===== SANITY CHECK =====
print("Shape demand_agg:", demand_agg.shape)
print(demand_agg.head(10))
print("\nMô tả thống kê d_bar / d_hat:")
print(demand_agg[["d_bar", "d_hat"]].describe())

C:\Users\USER\AppData\Local\Temp\ipykernel_25088\3855468875.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  b2b_agg["OrderMonth"] = b2b_agg["OrderedTimestamp"].dt.to_period("M")


Shape demand_agg: (1763, 4)
                         ShipToGroupID       d_bar        d_max       d_hat
0                           105695-002  134.691707   192.858195   58.166488
1                           118438-598  474.988263   916.489769  441.501506
2                           124624-000   95.059856   171.497252   76.437396
3                           128112-002   44.163454    61.779295   17.615841
4                           138789-000  100.016596   212.309713  112.293118
5                           150794-001   68.519253   113.816983   45.297730
6                           230001-001  826.089194  1409.251768  583.162574
7    ID_JV_Banten_Kota Cilegon_Cibeber    0.668009     4.123876    3.455867
8    ID_JV_Banten_Kota Cilegon_Cilegon    0.231137     0.342797    0.111659
9  ID_JV_Banten_Kota Cilegon_Citangkil    0.053710     0.236421    0.182711

Mô tả thống kê d_bar / d_hat:
             d_bar        d_hat
count  1763.000000  1763.000000
mean      3.899830     2.830026
std      

In [4]:
# ===== DIAGNOSE: vì sao node 230001-001 có demand cao bất thường =====

# Xem node này gộp từ bao nhiêu ShipToID gốc (mong đợi = 1 vì đây là dạng standalone, ko phải cluster ID_JV_...)
underlying_shipto = shipto_crosswalk[shipto_crosswalk["ShipToGroupID"] == "230001-001"]
print("Số ShipToID gốc thuộc node này:", underlying_shipto["ShipToID"].nunique())
print(underlying_shipto.head(10))

# Lấy toàn bộ order của node này (từ b2b_agg đã tính pallet ở bước 4.2)
node_orders = b2b_agg[b2b_agg["ShipToGroupID"] == "230001-001"]
print("\nTổng số dòng order:", len(node_orders))
print("Tổng kg:", node_orders["QtyOrderedInKG"].sum())
print("Tổng pallet:", node_orders["QtyInPallet"].sum())

# Demand theo tháng - xem tháng nào là peak (giải thích d_max=1409)
print("\nDemand theo tháng (pallet):")
print(node_orders.groupby("OrderMonth")["QtyInPallet"].sum())

# Top 10 sản phẩm đóng góp nhiều pallet nhất - check xem có sản phẩm nào KGPerPallet bất thường ko
print("\nTop 10 sản phẩm theo tổng pallet:")
top_products = node_orders.groupby("ProductID")["QtyInPallet"].sum().sort_values(ascending=False).head(10)
print(top_products)
print("\nKGPerPallet của các sản phẩm đó:")
print(product_master[product_master["ProductID"].isin(top_products.index)][["ProductID", "KGPerPallet"]])

# Xem thông tin channel của customer này - có phải national/modern trade lớn ko
print("\nThông tin customer:")
print(customer_master[customer_master["LocationID"].isin(underlying_shipto["ShipToID"])][["LocationID", "Channel", "SubChannel", "ChannelName"]])

Số ShipToID gốc thuộc node này: 1
     ShipToID ShipToGroupID
6  230001-001    230001-001

Tổng số dòng order: 1243
Tổng kg: 4236290.4036799995
Tổng pallet: 9086.981134259258

Demand theo tháng (pallet):
OrderMonth
2025-01     724.916667
2025-02     392.527778
2025-03     460.783333
2025-04     380.925926
2025-05     994.396970
2025-06    1097.440404
2025-07    1409.251768
2025-08    1365.042456
2025-09    1039.904356
2025-10     538.996907
2025-11     682.794571
Freq: M, Name: QtyInPallet, dtype: float64

Top 10 sản phẩm theo tổng pallet:
ProductID
06628    1724.527778
06629    1172.759259
06699    1120.750000
06630    1014.416667
06626     954.000000
06627     873.472222
06847     315.750000
06855     294.666667
07336     283.500000
07460     248.033333
Name: QtyInPallet, dtype: float64

KGPerPallet của các sản phẩm đó:
    ProductID  KGPerPallet
262     06628    486.00000
263     06626    493.34400
264     06855    380.16000
265     07460    411.32160
266     07336    363.13920
267 

## 4.3 Cluster Compactness Validation

Haversine distance between member `ShipToID`s and their cluster centroid, to support the methodology write-up (District-level grouping doesn't distort geography much).


In [6]:
# TODO: Haversine distance function
# TODO: compute per-cluster weighted_avg_dist_km
import numpy as np

# ===== BƯỚC 1: Hàm tính khoảng cách Haversine (km) giữa 2 điểm lat/long =====
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # bán kính Trái Đất (km)
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# ===== BƯỚC 2: Ghép tọa độ (CentroidLat/CentroidLong) của từng ShipToID vào shipto_crosswalk =====
cluster_coords = shipto_crosswalk.merge(
    customer_master[["LocationID", "CentroidLat", "CentroidLong"]],
    left_on="ShipToID", right_on="LocationID", how="left"
)

# ===== BƯỚC 3: Tính tâm cụm (centroid) = trung bình tọa độ các ShipToID trong cùng ShipToGroupID =====
cluster_centroids = cluster_coords.groupby("ShipToGroupID").agg(
    centroid_lat=("CentroidLat", "mean"),
    centroid_lon=("CentroidLong", "mean"),
    n_members=("ShipToID", "nunique")
).reset_index()

# ===== BƯỚC 4: Ghép tâm cụm ngược lại, tính khoảng cách từng ShipToID -> tâm cụm của nó =====
cluster_coords = cluster_coords.merge(cluster_centroids, on="ShipToGroupID", how="left")
cluster_coords["dist_to_centroid_km"] = haversine(
    cluster_coords["CentroidLat"], cluster_coords["CentroidLong"],
    cluster_coords["centroid_lat"], cluster_coords["centroid_lon"]
)

# ===== BƯỚC 5: Tính "bán kính cụm" (max/mean khoảng cách) cho mỗi ShipToGroupID =====
# Cụm standalone (1 member, ShipToGroupID = ShipToID) sẽ tự động ra bán kính = 0 - đúng như kỳ vọng
cluster_radius = cluster_coords.groupby("ShipToGroupID").agg(
    n_members=("ShipToID", "nunique"),
    max_radius_km=("dist_to_centroid_km", "max"),
    mean_radius_km=("dist_to_centroid_km", "mean")
).reset_index()

# ===== SANITY CHECK =====
print("Shape cluster_radius:", cluster_radius.shape)
print(cluster_radius.head(10))
print("\nMô tả thống kê bán kính cụm (km):")
print(cluster_radius[["max_radius_km", "mean_radius_km"]].describe())

# ===== Đếm số cụm có bán kính bất thường lớn (VD > 20km, coi như ngưỡng "không compact") =====
n_outlier_clusters = (cluster_radius["max_radius_km"] > 20).sum()
print(f"\nSố cụm có bán kính > 20km: {n_outlier_clusters} / {len(cluster_radius)}")

Shape cluster_radius: (1763, 4)
                         ShipToGroupID  n_members  max_radius_km  \
0                           105695-002          1       0.000000   
1                           118438-598          1       0.000000   
2                           124624-000          1       0.000000   
3                           128112-002          1       0.000000   
4                           138789-000          1       0.000000   
5                           150794-001          1       0.000000   
6                           230001-001          1       0.000000   
7    ID_JV_Banten_Kota Cilegon_Cibeber          8       0.693171   
8    ID_JV_Banten_Kota Cilegon_Cilegon          4       0.671852   
9  ID_JV_Banten_Kota Cilegon_Citangkil          8       1.608656   

   mean_radius_km  
0        0.000000  
1        0.000000  
2        0.000000  
3        0.000000  
4        0.000000  
5        0.000000  
6        0.000000  
7        0.693171  
8        0.335926  
9        0.889924  

## 4.4 Facility Classification (Fixed / Decision / Excluded)

| Role | FacilityType | Count |
|---|---|---|
| **FIXED** | `NDC` | 1 |
| **DECISION** | `DEPO`, `RDC`, `DC Direct`, `DC Satellite` | 21 |
| **EXCLUDED** | `FC` (B2C only); `Instant Hub` (= `MFC` theo `Network_Map.md` chính thức của CEL: `DC Direct → MFC → B2C Customers`, chỉ phục vụ B2C và ko phát sinh chi phí do Paragon chi trả) | 5 |

**Tổng facility_model dùng cho MILP (FIXED + DECISION): 22**

> Instant Hub ban đầu bị nghi ngờ do 2 phát hiện thực nghiệm: (1) zero node phụ thuộc duy nhất vào nó trong coverage check B2B, (2) hoàn toàn vắng mặt như Origin hợp lệ trong bảng `CostLastMileB2B` (Unconstrained/Greenfield). Việc xác nhận Instant Hub = MFC (node B2C-only theo network map chính thức) giải thích trọn vẹn cả 2 phát hiện trên — nó chưa từng là facility B2B.


In [ ]:
# ===== BƯỚC 1: Định nghĩa 3 nhóm facility theo vai trò trong model =====
# FIXED: bắt buộc luôn mở (NDC là node gốc nhận hàng từ factory, ko phải biến quyết định mở/đóng)
FIXED_TYPES = ["NDC"]

# DECISION: facility model sẽ quyết định mở hay đóng (biến nhị phân y_j trong MILP)
DECISION_TYPES = ["DEPO", "RDC", "DC Direct", "DC Satellite"]

# EXCLUDED: ngoài scope B2B distribution của thesis này (chỉ phục vụ B2C)
# - FC: chỉ phục vụ B2C
# - Instant Hub: = MFC theo Network_Map.md chính thức của CEL (DC Direct -> MFC -> B2C Customers),
#   cũng chỉ phục vụ B2C và ko phát sinh chi phí do Paragon chi trả (out of scope cho model B2B này)
EXCLUDED_TYPES = ["FC", "Instant Hub"]

# ===== BƯỚC 2: Gắn nhãn vai trò cho từng facility dựa theo FacilityType =====
def classify_facility(facility_type):
    if facility_type in FIXED_TYPES:
        return "FIXED"
    elif facility_type in DECISION_TYPES:
        return "DECISION"
    elif facility_type in EXCLUDED_TYPES:
        return "EXCLUDED"
    else:
        return "UNKNOWN"  # bắt lỗi nếu có FacilityType nào chưa được xếp loại

facility_master["FacilityRole"] = facility_master["FacilityType"].apply(classify_facility)

# ===== BƯỚC 3: Tách riêng từng nhóm + tạo bảng facility_model dùng cho MILP (loại EXCLUDED) =====
facility_fixed = facility_master[facility_master["FacilityRole"] == "FIXED"].copy()
facility_decision = facility_master[facility_master["FacilityRole"] == "DECISION"].copy()
facility_excluded = facility_master[facility_master["FacilityRole"] == "EXCLUDED"].copy()
facility_model = facility_master[facility_master["FacilityRole"] != "EXCLUDED"].copy()

# ===== SANITY CHECK =====
print("Phân bố FacilityRole:")
print(facility_master["FacilityRole"].value_counts())

print("\nCó FacilityType nào bị xếp UNKNOWN ko (nếu có nghĩa thiếu xử lý):")
print(facility_master[facility_master["FacilityRole"] == "UNKNOWN"][["LocationID", "FacilityType"]])

print("\nSố lượng theo từng nhóm:")
print("FIXED:", facility_fixed.shape[0])
print("DECISION:", facility_decision.shape[0])
print("EXCLUDED:", facility_excluded.shape[0])
print("Tổng dùng cho model (FIXED + DECISION):", facility_model.shape[0])

print("\nHead(10) facility_model:")
print(facility_model.head(10))

## 4.5 Distance & Coverage Feasibility

$Dist_{jk}$ via Haversine (interim, pending OSRM road-distance file). Coverage constraint uses $D_{max}=150$ km (Java B2B SLA, Paragon's Assumption Doc §4.7).


In [ ]:
# ===== BƯỚC 1: Load bảng cost THẬT (Unconstrained/Greenfield - free-flow, KHÔNG bị force bởi Share*) =====
# LƯU Ý QUAN TRỌNG: KHÔNG dùng ENO_CostLastMileB2B.parquet ở scenario Baseline 2025 (Constrained).
# Bảng Constrained chỉ phản ánh route đã từng xảy ra thực tế trong quá khứ (bị ép 100% qua
# ShareLastMileB2B), KHÔNG phải chi phí free-choice cho toàn bộ facility-cluster pair.
# Dùng bảng Unconstrained/Greenfield để model có đầy đủ lựa chọn định tuyến (freely flow).
cost_lastmile = pd.read_parquet(os.path.join(RAW_DATA_PATH, "ENO_CostLastMileB2B_Unconstaint.parquet"))

# ===== BƯỚC 2: Chỉ giữ cost record có Origin nằm trong facility_model (FIXED+DECISION, 22 facility) =====
# FC, Instant Hub (=MFC) (EXCLUDED) chỉ phục vụ B2C nên cost từ các facility này ko liên quan tới network B2B đang model
valid_facility_ids = set(facility_model["LocationID"])
cost_lastmile_scoped = cost_lastmile[cost_lastmile["OriginID"].isin(valid_facility_ids)].copy()

# ===== BƯỚC 3: Với mỗi demand node, đếm có bao nhiêu facility CÓ THỂ phục vụ (có sẵn cost record) =====
coverage_count = cost_lastmile_scoped.groupby("DestinationID")["OriginID"].nunique().reset_index()
coverage_count.columns = ["ShipToGroupID", "n_facilities_covering"]

# ===== BƯỚC 4: Ghép với demand_agg để biết node nào KHÔNG có facility nào cover (0 = infeasible) =====
demand_coverage = demand_agg[["ShipToGroupID", "d_bar", "d_hat"]].merge(
    coverage_count, on="ShipToGroupID", how="left"
)
demand_coverage["n_facilities_covering"] = demand_coverage["n_facilities_covering"].fillna(0).astype(int)

# ===== SANITY CHECK =====
print("Shape demand_coverage:", demand_coverage.shape)
print(demand_coverage.head(10))

print("\nPhân bố số facility cover mỗi node:")
print(demand_coverage["n_facilities_covering"].value_counts().sort_index())

print("\nCoverage trung bình (facility option / node):", demand_coverage["n_facilities_covering"].mean())

# ===== BƯỚC 5: Xác định node KHÔNG có facility nào cover - nghiêm trọng nếu tồn tại =====
uncovered_nodes = demand_coverage[demand_coverage["n_facilities_covering"] == 0]
print(f"\nSố node KHÔNG có facility nào cover: {len(uncovered_nodes)} / {len(demand_coverage)}")
if len(uncovered_nodes) > 0:
    print("Các node này sẽ khiến model INFEASIBLE nếu bắt buộc đáp ứng đủ demand:")
    print(uncovered_nodes.head(10))
    print("\nTổng d_bar bị ảnh hưởng:", uncovered_nodes["d_bar"].sum())

In [12]:
# ===== CHECK: facility "duy nhất" cover 1 node có đúng là facility GẦN NHẤT không =====

# Lấy thử 1 node chỉ có 1 lựa chọn để kiểm tra (VD node đầu tiên trong danh sách)
sample_node = "105695-002"

# Lấy tọa độ tâm cụm của node này (đã tính ở bước 4.3)
sample_coord = cluster_centroids[cluster_centroids["ShipToGroupID"] == sample_node]

# Tính khoảng cách từ node này tới TẤT CẢ 24 facility trong facility_model (không chỉ facility đang cover)
facility_model["dist_to_sample_km"] = haversine(
    facility_model["Latitude"], facility_model["Longitude"],
    sample_coord["centroid_lat"].values[0], sample_coord["centroid_lon"].values[0]
)

print("Xếp hạng khoảng cách từ node này tới TẤT CẢ facility:")
print(facility_model[["LocationID", "FacilityName", "FacilityType", "dist_to_sample_km"]].sort_values("dist_to_sample_km").head(10))

print("\nFacility THỰC SỰ có cost record (theo bảng CEL) tới node này:")
print(cost_lastmile_scoped[cost_lastmile_scoped["DestinationID"] == sample_node])

Xếp hạng khoảng cách từ node này tới TẤT CẢ facility:
   LocationID              FacilityName  FacilityType  dist_to_sample_km
0         D09           DC Direct Bogor     DC Direct           7.407398
15        D54     Instant Hub Fatmawati   Instant Hub          25.866089
17     DEPO04    Depo Bekasi - DC Bogor          DEPO          28.960587
25     DEPO03  Depo Jakarta - DC Banten          DEPO          35.247879
14        D49     Instant Hub Pulo Asem   Instant Hub          37.475494
12    MWH_D35     DC Satellite Sukabumi  DC Satellite          46.764927
26        NDC              NDC Jatake 6           NDC          51.565794
13    MWH_D36          DC Direct Banten     DC Direct          69.154784
7     MWH_D04               RDC Bandung           RDC          97.429590
11    MWH_D27  DC Satellite Tasikmalaya  DC Satellite         169.423252

Facility THỰC SỰ có cost record (theo bảng CEL) tới node này:
    OriginID DestinationID  CostPerPallet
143      D09    105695-002  316867.648

In [14]:
# ===== CHECK: facility nào đang là "nhà cung cấp duy nhất" - đóng nó = mất bao nhiêu node =====
single_option_nodes = demand_coverage[demand_coverage["n_facilities_covering"] == 1]["ShipToGroupID"]
sole_providers = cost_lastmile_scoped[cost_lastmile_scoped["DestinationID"].isin(single_option_nodes)]

sole_provider_load = sole_providers.groupby("OriginID")["DestinationID"].nunique().sort_values(ascending=False).reset_index()
sole_provider_load.columns = ["LocationID", "n_nodes_depend_solely"]
sole_provider_load = sole_provider_load.merge(
    facility_model[["LocationID", "FacilityName", "FacilityType", "FacilityRole"]], on="LocationID", how="left"
)

print(sole_provider_load)

# Facility DECISION nào KHÔNG xuất hiện trong danh sách trên = ko phụ thuộc node nào -> tự do đóng thoải mái
free_to_close = facility_decision[~facility_decision["LocationID"].isin(sole_provider_load["LocationID"])]
print("\nFacility DECISION hoàn toàn tự do đóng (ko sole-provider cho node nào):")
print(free_to_close[["LocationID", "FacilityName", "FacilityType"]])

   LocationID  n_nodes_depend_solely                         FacilityName  \
0         D26                    133                   DC Direct Surabaya   
1      DEPO18                    129            Depo Yogyakarta - DC Solo   
2     MWH_D12                    128                           RDC Jember   
3         D25                    127                       DC Direct Solo   
4     MWH_D04                    107                          RDC Bandung   
5     MWH_D21                    103              DC Satellite Purwokerto   
6     MWH_D27                    103             DC Satellite Tasikmalaya   
7         D10                     96                          RDC Cirebon   
8         D30                     93                           RDC Kediri   
9      DEPO12                     83              Depo Pati - DC Semarang   
10    MWH_D36                     83                     DC Direct Banten   
11    MWH_D23                     73                   DC Direct Semarang   

In [16]:
# ===== CHECK: rate có ổn định hơn khi tách theo từng facility (OriginID) không =====
print(cost_with_dist.groupby("OriginID")["rate_per_km"].describe())

          count          mean           std           min           25%  \
OriginID                                                                  
D09        76.0  36397.941507   7110.415173  27538.854908  31447.455165   
D10       101.0  42623.558063   5914.207932  33055.769543  37992.182935   
D25       143.0  29498.021073   3387.662441  22110.522409  27122.048077   
D26       152.0  10233.889165   1314.985707   8025.426620   9164.547091   
D30       103.0  43864.735322   4855.464228  36390.459565  40377.714645   
DEPO02     55.0  49624.183364  10747.717318  37678.112489  41823.885695   
DEPO03     66.0  31308.270109   5286.034924  24710.421599  28418.625177   
DEPO04     39.0  52853.824837   6460.914444  44678.718464  49010.130645   
DEPO07     60.0  36491.853015   3994.719265  28468.977470  33742.591458   
DEPO08     37.0  39530.485753   5518.041911  30081.526779  35826.466006   
DEPO12     83.0  47121.462064   6407.141129  36112.460518  42149.012140   
DEPO13     38.0  12800.09

## 4.6 Cost Parameters

Load và filter 4 bảng cost. **Mỗi bảng lấy đúng scenario nguồn** (xem lý do chọn Unconstrained/Greenfield ở Section 4.5):

| File | Scenario nguồn | Lý do |
|---|---|---|
| `ENO_CostLastMileB2B_Unconstaint.parquet` | Unconstrained/Greenfield | Free-flow cost, ko bị force bởi `Share*`; bảng Baseline chỉ phản ánh route lịch sử |
| `3_GF_Unconstrained/ENO_CostTransfer.parquet` | Unconstrained/Greenfield | Tương tự — free-flow transfer cost giữa các facility |
| `0,1_Baseline2025/ENO_CostSupply.parquet` | Baseline | Scenario-invariant (toàn bộ `CostPerPallet` = 0, Factory→NDC) |
| `0,1_Baseline2025/ENO_DOS.parquet` | Baseline | Ko phải bảng O-D nên ko bị ảnh hưởng bởi Constrained/Unconstrained |

Cả 2 bảng Unconstrained đều được filter chặt về đúng 22 facility trong `facility_model` — **KHÔNG** lấy thêm các candidate site Greenfield mới (giữ nguyên scope facility hiện hữu của thesis).


In [ ]:
# ===== BƯỚC 1: Load cost tables - MỖI bảng lấy đúng scenario nguồn (xem bảng ở markdown trên) =====
cost_lastmile_b2b = pd.read_parquet(os.path.join(RAW_DATA_PATH, "ENO_CostLastMileB2B_Unconstaint.parquet"))
cost_transfer = pd.read_parquet(os.path.join(RAW_DATA_PATH, "3_GF_Unconstrained", "ENO_CostTransfer.parquet"))
cost_supply = pd.read_parquet(os.path.join(RAW_DATA_PATH, "0,1_Baseline2025", "ENO_CostSupply.parquet"))
dos = pd.read_parquet(os.path.join(RAW_DATA_PATH, "0,1_Baseline2025", "ENO_DOS.parquet"))

# ===== BƯỚC 2: Filter về đúng scope facility_model (22 facility hiện hữu, KHÔNG lấy candidate site Greenfield mới) =====
valid_facility_ids = set(facility_model["LocationID"])
valid_shipto_ids = set(demand_agg["ShipToGroupID"])

# CostLastMileB2B: Origin phải trong facility_model, Destination phải trong demand_agg (Java ShipToGroupID)
cost_lastmile_final = cost_lastmile_b2b[
    cost_lastmile_b2b["OriginID"].isin(valid_facility_ids) &
    cost_lastmile_b2b["DestinationID"].isin(valid_shipto_ids)
].copy()

# CostTransfer: cả Origin và Destination đều phải trong facility_model
transfer_final = cost_transfer[
    cost_transfer["OriginID"].isin(valid_facility_ids) &
    cost_transfer["DestinationID"].isin(valid_facility_ids)
].copy()

# CostSupply: giữ nguyên bản Baseline (chỉ có Factory -> NDC, ko cần filter theo facility_model)
supply_baseline = cost_supply.copy()

# ===== SANITY CHECK =====
print("cost_lastmile_final shape:", cost_lastmile_final.shape)
print(cost_lastmile_final.head(10))

print("\ntransfer_final shape:", transfer_final.shape)
print(transfer_final.head(10))

print("\nsupply_baseline shape:", supply_baseline.shape)
print(supply_baseline.head(10))

print("\ndos shape:", dos.shape)
print(dos.head(10))

# ===== Coverage check trên bảng đã filter =====
coverage_check = cost_lastmile_final.groupby("DestinationID")["OriginID"].nunique()
print("\nCoverage trung bình (facility option / node):", coverage_check.mean())
print("Số node ko có facility nào cover:", (~demand_agg["ShipToGroupID"].isin(coverage_check.index)).sum())

# TODO: DOS chưa filter theo facility_model - cần confirm có cần lọc LocationID trước khi đưa vào model ko

---
# 5. Solution Method

## 5.1 Robust Optimization Reformulation

> ✍️ **Write here:** explain the Bertsimas & Sim (2004) budget-of-uncertainty approach — how $\bar d_k$, $\hat d_k$, and $\Gamma$ combine into the robust counterpart constraint used in Section 3.2. State the chosen $\Gamma$ value or the sensitivity range tested, and why.

## 5.2 AUGMECON2

> ✍️ **Write here:** explain the augmented ε-constraint method with bypass (Mavrotas & Florios, 2013) — why it's used instead of plain ε-constraint or weighted sum, and how the grid/step size for $f_2$, $f_3$ is chosen.

## 5.3 TOPSIS

> ✍️ **Write here:** explain how the Pareto set is ranked post-hoc under different manager priority scenarios (cost-priority / environment-priority / balanced), and the criteria weights used.


---
# 6. Implementation

## 6.1 Pyomo Model


In [ ]:
# TODO: build Pyomo model — sets, parameters, variables, objectives, constraints


## 6.2 AUGMECON2 Solve Loop


In [ ]:
# TODO: AUGMECON2 implementation


## 6.3 TOPSIS Ranking


In [ ]:
# TODO: TOPSIS ranking on Pareto set


---
# 7. Results & Discussion

> ✍️ **Write here** once the model runs: Pareto front summary, selected scenarios from TOPSIS, sensitivity to $\Gamma$, network configuration decisions (facility open/close), and comparison to the current (baseline) network.

*(placeholder — fill in after running Section 6)*
